# Soluciones · 05 Validación y calidad de la medida

Soluciones resueltas de los ejercicios del [cuaderno 05](../../notebooks/05_validacion_calidad.ipynb). Intenta antes cada ejercicio por tu cuenta.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "vnacourse[validation] @ git+https://github.com/telmomm/nanovna-measurement-course"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import skrf as rf
from numpy.lib.stride_tricks import sliding_window_view
from rfmeasurement.domain import (
    Measurement,
    MeasurementContext,
    MetadataConfidence,
    ValidationResult,
    ValidationStatus,
)
from rfmeasurement.validation import ValidationRule

from vnacourse import calibration, plotting
from vnacourse.data import load_repetitions

plotting.use_course_style()

f_hz, short_reps = load_repetitions("standards", "short")
_, open_reps = load_repetitions("standards", "open")
_, load_reps = load_repetitions("standards", "load")
_, resistor_reps = load_repetitions("dut", "load_75ohm")
f_mhz = f_hz / 1e6
standards = (short_reps, open_reps, load_reps)

## 1. ¿Media o mediana?

La mediana de números complejos se toma sobre la parte real y la imaginaria por separado. Todo lo demás es como en la sección 4: calibra con las repeticiones 1–5 y prueba con las repeticiones 6–10 del abierto.

In [ ]:
def mean(repetitions):
    return repetitions.mean(axis=0)


def median(repetitions):
    return np.median(repetitions.real, axis=0) + 1j * np.median(repetitions.imag, axis=0)


fig, ax = plt.subplots()
for name, combine in (("media", mean), ("mediana", median)):
    terms = calibration.solve_error_terms([combine(reps[:5]) for reps in standards])
    distance = np.abs(calibration.correct(combine(open_reps[5:]), terms) - 1)
    ax.semilogy(f_mhz, distance, label=name, linewidth=1.0)
    print(
        f"{name:<7} típica {np.median(distance):.4f}   peor {distance.max():.4f}   "
        f"puntos más allá de 0.01: {np.sum(distance > 0.01)}"
    )
ax.set(xlabel="frecuencia (MHz)", ylabel="$|\\Gamma - 1|$ del abierto calibrado (escala log.)")
ax.set_title("Combinar repeticiones con la media o con la mediana")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2);

La mediana gana con claridad en los picos: el peor punto baja de 0.040 a 0.019 y el número de puntos a más de 0.01 del valor ideal cae de 42 a 6. La distancia típica también mejora ligeramente.

La media de cinco valores se desplaza un quinto del tamaño de un valor malo; la mediana lo ignora mientras menos de la mitad de los valores sean malos. El precio: con ruido limpio y normal, la mediana es algo más ruidosa que la media. Cuando los datos tienen valores atípicos, como aquí, es un buen intercambio. Mejor todavía es averiguar por qué aparecen los picos y evitarlos en la adquisición.

## 2. Incertidumbre en toda la banda

Diez calibraciones, una por repetición de los estándares, cada una aplicada al barrido medio de la resistencia. Su dispersión a cada frecuencia es la contribución de la repetibilidad de la calibración al $\Gamma$ corregido.

In [ ]:
resistor_mean = resistor_reps.mean(axis=0)
corrected = np.array(
    [
        calibration.correct(
            resistor_mean,
            calibration.solve_error_terms([short_reps[i], open_reps[i], load_reps[i]]),
        )
        for i in range(len(open_reps))
    ]
)
scatter = np.sqrt(corrected.real.var(axis=0, ddof=1) + corrected.imag.var(axis=0, ddof=1))

fig, ax = plt.subplots()
ax.semilogy(f_mhz, scatter, linewidth=1.0)
ax.set(xlabel="frecuencia (MHz)", ylabel="desviación típica del Γ corregido (escala log.)")
ax.set_title("Resistencia de 75 Ω: dispersión entre diez calibraciones")

for label, band in (
    ("por debajo de 150 MHz", f_hz < 150e6),
    ("de 150 a 500 MHz", (f_hz >= 150e6) & (f_hz < 500e6)),
    ("por encima de 1 GHz", f_hz > 1e9),
):
    print(f"{label:<22} dispersión típica {np.median(scatter[band]):.5f}")
print(f"peor punto: {scatter.max():.4f} a {f_mhz[scatter.argmax()]:.2f} MHz")

No es uniforme. La calibración se repite mejor entre 150 MHz y 500 MHz (unos 0.0005), es aproximadamente el doble de ruidosa por encima de 1 GHz, y también es peor por debajo de 150 MHz, la banda en la que la respuesta en bruto del instrumento está lejos de lo ideal. Los picos de la sección 2 se trasladan al resultado calibrado, y el primer punto del barrido es el peor.

Compara el nivel con el de la sección 2: los estándares se dispersan unos 0.003, pero la resistencia corregida solo unos 0.0007. La dispersión del cortocircuito y del abierto entra en el resultado multiplicada por el pequeño $|\Gamma| = 0.2$ de la resistencia; un dispositivo cerca del borde de la carta la heredaría por completo. La incertidumbre de un resultado calibrado depende tanto del dispositivo como de la calibración.

## 3. Escribe una regla

Un pico es un punto que se aparta de sus vecinos. Compara cada punto de $|\Gamma|$ con la mediana de una ventana de siete puntos centrada en él; la mediana sigue la tendencia suave e ignora un único valor atípico.

In [ ]:
class SpikesRule(ValidationRule):
    """Avisa cuando hay puntos que se apartan de la mediana local de |Γ| más que un umbral."""

    identifier = "quality.spikes"
    description = "Ningún punto se aparta de la mediana de sus vecinos más que el umbral."

    def __init__(self, threshold=0.02, window=7):
        self.threshold = threshold
        self.window = window

    def _evaluate(self, measurement):
        magnitude = np.abs(measurement.data.s[:, 0, 0])
        padded = np.pad(magnitude, self.window // 2, mode="edge")
        local_median = np.median(sliding_window_view(padded, self.window), axis=1)
        spikes = np.flatnonzero(np.abs(magnitude - local_median) > self.threshold)
        return ValidationResult(
            rule_id=self.identifier,
            status=ValidationStatus.WARNING if len(spikes) else ValidationStatus.PASS,
            description=self.description,
            evidence={
                "threshold": self.threshold,
                "n_spikes": len(spikes),
                "frequency_indices": spikes.tolist(),
            },
            explanation="Desviaciones aisladas de la tendencia local." if len(spikes) else None,
        )


frequency = rf.Frequency.from_f(f_hz, unit="Hz")
rule = SpikesRule()
for repetition in (0, 8):
    sweep = Measurement(
        data=rf.Network(frequency=frequency, s=open_reps[repetition], z0=50),
        context=MeasurementContext(
            dut="abierto", instrument="NanoVNA-F V2", calibration="ninguna (en bruto)"
        ),
    )
    result = rule.evaluate(sweep)
    indices = result.evidence["frequency_indices"]
    print(f"repetición {repetition + 1}: {result.status.value}, picos en los puntos {indices}")
    print("   frecuencias (MHz):", np.round(f_mhz[indices], 1))

La regla encuentra tres picos genuinos en la repetición 9, en los puntos 301, 502 y 904: todos justo después de un múltiplo de 100, como encontró la sección 2 a partir de la estadística de diez barridos. Aquí basta un único barrido.

También marca los puntos 1 a 5 en ambas repeticiones. Esos no son picos. En el extremo inferior de la banda, la respuesta en bruto de este instrumento cambia de verdad bruscamente de un punto al siguiente, y una regla que busca «desviaciones respecto a los vecinos» no puede distinguir un cambio real rápido de un valor atípico. Por eso la regla devuelve un `warning` con su evidencia y no un `fail`: señala lugares que mirar, y una persona decide qué son. Una regla ajustada hasta no dar nunca una falsa alarma en este conjunto de datos sería una regla ajustada a este conjunto de datos.

## 4. Tu propio contexto

No hay una única respuesta correcta. Esta es una para una medida imaginaria de un electrodo cutáneo, con el razonamiento de cada nivel de confianza.

In [ ]:
context = MeasurementContext(
    dut="Electrodo de Ag/AgCl, lote 12, muestra 3",
    operator="Un estudiante",
    instrument="NanoVNA-F V2, número de serie 0421",
    calibration="SOL al final del cable de prueba, definiciones del kit del proveedor",
    cables="30 cm de RG-316, SMA",
    fixture="soporte de electrodo a medida, revisión B",
    temperature_c=23.5,
    averaging=10,
    frequency_range_hz=(50e3, 1.45e9),
    confidence={
        "dut": MetadataConfidence.SPECIFIED,
        "instrument": MetadataConfidence.SPECIFIED,
        "calibration": MetadataConfidence.SPECIFIED,
        "cables": MetadataConfidence.SPECIFIED,
        "temperature_c": MetadataConfidence.MEASURED,
        "frequency_range_hz": MetadataConfidence.MEASURED,
        "averaging": MetadataConfidence.MEASURED,
    },
)

for field in context.__dataclass_fields__:
    value = getattr(context, field)
    if field == "confidence":
        continue
    level = context.confidence.get(field)
    print(
        f"{field:<20} {str(value) if value not in (None, {}) else '— desconocido':<70} "
        f"{level.value if level else ''}"
    )

- **Medido** (*measured*): lo que leyó un instrumento durante la sesión (la temperatura de un termómetro en la mesa, el intervalo de frecuencias y el promediado del propio barrido).
- **Especificado** (*specified*): lo que te dijeron y anotaste (la etiqueta de la pieza, el número de serie del instrumento, las definiciones del kit del proveedor). Te fías de un documento, no de una lectura.
- **Estimado o supuesto** (*estimated*, *assumed*): todo lo que no comprobaste. Si no hubieras medido la temperatura y escribieras «unos 23 °C porque el laboratorio tiene aire acondicionado», eso es *supuesto*, y decirlo es más útil que un número a secas.
- **Desconocido:** se deja vacío. Humedad, presión, potencia de RF y ancho de banda de FI están vacíos aquí; un campo vacío es una respuesta honesta; uno adivinado, no.

Los campos que con más frecuencia resultan importar después, y que más a menudo faltan, son la calibración (dónde y con qué), los cables y el utillaje, y la fecha.